# Tiny ImageNet-C sơ bộ trên Kaggle

Dùng Dataset hiện có: 200 lớp, JPEG 64×64, tên `test_*.JPEG`. **1 panel, 3 kịch bản, W=0/4/16, source/norm/Tent/SAR, reset none/all: 72 cặp, 144 lượt Q.** Prefix=512, washout tối đa=512, Q=512, batch size=32.

Mô hình: ResNet50 ImageNet-1K V1 pretrained, chỉ giữ 200 logits tương ứng các synset của Dataset **trước** khi tính softmax, entropy và loss. Dùng transform của trọng số ImageNet (resize 256, crop 224). Không huấn luyện mô hình nguồn Tiny; kết quả là exploratory transfer trên Tiny, không phải kết quả Stage B ImageNet-C hay baseline Tiny đã được huấn luyện.

**Trước Run All:** bật GPU + Internet; Add Input Dataset nguồn mới từ `streaming-tta-kaggle-tiny-source.zip`; giữ Dataset ảnh hiện có. Có thể giữ Input nguồn cũ: notebook chỉ chọn nguồn chứa `scripts/tiny_imagenetc.py`. Xem `research/kaggle_tiny_preliminary.md`.

In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
from pathlib import Path
import shutil, subprocess, sys, json, zipfile, tempfile

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "streaming-tta-tiny-state-memory"
sources = [
    p.parent.parent for p in INPUT.rglob("tiny_imagenetc.py")
    if p.parent.name == "scripts"
    and (p.parent.parent / "src/historytta").is_dir()
    and (p.parent.parent / "configs/kaggle_tiny_preliminary.yaml").is_file()
]
if not REPO.exists():
    if len(sources) == 1:
        shutil.copytree(sources[0], REPO)
    elif not sources:
        bundles = list(INPUT.rglob("streaming-tta-kaggle-tiny-source.zip"))
        if len(bundles) != 1:
            raise RuntimeError("Add Input the NEW source bundle streaming-tta-kaggle-tiny-source.zip.")
        with tempfile.TemporaryDirectory(prefix="tiny_source_", dir=WORK) as temp:
            temp = Path(temp)
            with zipfile.ZipFile(bundles[0]) as archive:
                if sum(i.file_size for i in archive.infolist()) > 100_000_000:
                    raise RuntimeError("Unexpectedly large source bundle.")
                expected_root = temp / "streaming-tta-state-memory"
                for info in archive.infolist():
                    if not (temp / info.filename).resolve().is_relative_to(expected_root.resolve()):
                        raise RuntimeError("Unexpected ZIP path: " + info.filename)
                    if (info.external_attr >> 16) & 0o170000 == 0o120000:
                        raise RuntimeError("Source symlinks are not supported.")
                archive.extractall(temp)
            shutil.copytree(expected_root, REPO)
    else:
        raise RuntimeError("Multiple NEW source Datasets found; keep exactly one.")
if not (REPO / "scripts/tiny_imagenetc.py").is_file():
    raise RuntimeError("Tiny source is missing. Attach the new bundle and use a fresh session.")
os.chdir(REPO)
print("Tiny repository:", REPO)


Tiny repository: /kaggle/working/streaming-tta-tiny-state-memory


## 1. Tự tìm dữ liệu

Nguồn mặc định là Dataset bạn đang dùng. Nếu đổi Dataset, sửa `DATA_SOURCE_URL`. Không sửa batch size, W hay các counts để bỏ qua lỗi. Sampler giữ hash split pilot/reserve, không lấy ảnh reserve khi thiếu pilot.

In [2]:
DATA_SOURCE_URL = "https://www.kaggle.com/datasets/husnifdu/imagenet-c"
domains = ["gaussian_noise", "brightness", "defocus_blur"]
roots = [
    p.parent for p in INPUT.rglob("gaussian_noise")
    if all((p.parent / d / "5").is_dir() for d in domains)
]
if len(roots) != 1:
    print("Candidate image roots:", roots)
    raise RuntimeError("Attach one Tiny ImageNet-C Dataset containing all three corruption/5 directories.")
DATA_ROOT = roots[0]
CONFIG = REPO / "configs/kaggle_tiny_preliminary.yaml"
CLASS_INDEX = REPO / "datasets/metadata/imagenet_class_index.json"
DOWNLOAD_CLASS_INDEX = True
MAX_HOURS = 10.0
EXPORT_DIR = WORK / "kaggle_tiny_exports"

print("DATA_ROOT:", DATA_ROOT)
for domain in domains:
    variant = DATA_ROOT / domain / "5"
    count = sum(p.is_dir() for p in variant.iterdir())
    print(domain, "classes:", count)
    if count != 200:
        raise RuntimeError("This notebook requires exactly 200 Tiny classes in every variant.")


DATA_ROOT: /kaggle/input/datasets/husnifdu/imagenet-c/ImageNet-C
gaussian_noise classes: 200
brightness classes: 200
defocus_blur classes: 200


## 2. Kiểm tra môi trường

Giữ Torch/torchvision sẵn có của Kaggle. Internet dùng để lấy mapping ImageNet 35 KB, trọng số ResNet50 V1 và gói nhỏ còn thiếu. Notebook không tải archive ảnh.

In [3]:
import importlib.util
for module in ["torch", "torchvision"]:
    if importlib.util.find_spec(module) is None:
        raise RuntimeError("Use a Kaggle GPU image with " + module)
packages = {"yaml": "PyYAML", "numpy": "numpy", "pandas": "pandas", "scipy": "scipy",
            "matplotlib": "matplotlib", "PIL": "Pillow", "pytest": "pytest", "nbformat": "nbformat"}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
import torch, torchvision
if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU in Kaggle Settings.")
print("Torch:", torch.__version__, "torchvision:", torchvision.__version__)
print("CUDA:", torch.version.cuda, "GPU:", torch.cuda.get_device_name(0))
probe = torch.ones((8, 8), device="cuda")
print("CUDA smoke:", (probe @ probe).mean().item())
del probe
torch.cuda.empty_cache()


Torch: 2.10.0+cu128 torchvision: 0.25.0+cu128
CUDA: 12.8 GPU: Tesla T4
CUDA smoke: 8.0


## 3. Kế hoạch và tests nhỏ

Tests dùng fixtures CPU, không tải trọng số ResNet50. Đây là kiểm tra code, không phải kết quả Tiny thật.

In [4]:
def helper(*arguments):
    subprocess.run([sys.executable, "-u", "scripts/kaggle_preliminary.py",
                    *map(str, arguments), "--config", str(CONFIG)], cwd=REPO, check=True)

helper("plan")
subprocess.run([sys.executable, "-m", "pytest", "-q",
                "tests/test_tiny_imagenetc.py", "tests/test_adapters.py",
                "tests/test_protocol.py", "tests/test_evaluate.py"], cwd=REPO, check=True)


{
  "mode": "plan_only",
  "config": {
    "experiment_id": "tiny_imagenetc_kaggle_preliminary_v1",
    "stage": "TINY",
    "dataset": "Tiny ImageNet-C",
    "dataset_profile": "tiny_imagenet_c",
    "output": "results/tiny_imagenetc_kaggle_preliminary_v1",
    "manifest_index": "datasets/processed/tiny-imagenet-c-kaggle-preliminary/index.json",
    "class_index": "datasets/metadata/imagenet_class_index.json",
    "backbone": "resnet50",
    "weights": "IMAGENET1K_V1",
    "classifier": "imagenet_200_class_projection",
    "expected_classes": 200,
    "device": "cuda",
    "threads": 2,
    "methods": [
      "source",
      "norm",
      "tent",
      "sar_complete"
    ],
    "washout_batches": [
      0,
      4,
      16
    ],
    "interventions": [
      "none",
      "all"
    ],
    "adaptation": {
      "batch_size": 32,
      "lr": 0.00025,
      "momentum": 0.9,
      "reset_threshold": 0.2,
      "sar_margin": 2.1193269466192146
    },
    "save_junctions": false,
    "int

CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '-q', 'tests/test_tiny_imagenetc.py', 'tests/test_adapters.py', 'tests/test_protocol.py', 'tests/test_evaluate.py'], returncode=0)

## 4. Tạo manifest Tiny

Mapping nhãn 0..199 được tạo theo thứ tự synset và ánh xạ sang chỉ số ImageNet gốc. Ba corruption phải cùng ID và lớp. JPEG được chọn phải 64×64. Cần **1536 ID pilot**, tương ứng 4608 file JPEG qua ba corruption. Mirror được ghi rõ nguồn khai báo và hash ảnh được chọn; không tuyên bố đã kiểm MD5 archive chính thức.

In [5]:
arguments = ["prepare", "--data-root", DATA_ROOT, "--source-url", DATA_SOURCE_URL,
             "--class-index", CLASS_INDEX]
if DOWNLOAD_CLASS_INDEX:
    arguments.append("--download-class-index")
helper(*arguments)


{
  "index": "/kaggle/working/streaming-tta-tiny-state-memory/datasets/processed/tiny-imagenet-c-kaggle-preliminary/index.json",
  "dataset": "Tiny ImageNet-C",
  "classes": 200,
  "panels": 3,
  "unique_base_images": 1536,
  "available_pilot_ids": 2060,
  "classifier": "imagenet_200_class_projection",
  "provenance_mode": "declared_attached_dataset"
}


## 5. Chạy, audit và xuất ZIP

Mọi adaptation loss được tính trên 200 logits. SAR margin=0.4×log(200). Runner chỉ dùng GPU đầu tiên. Cap subprocess 10 giờ không bao gồm chuẩn bị dữ liệu, tests hoặc render hình.

Kết quả hoàn chỉnh cần run_status=complete, audit passed và tất cả controls đạt. `finally` xuất diagnostics cả khi runner lỗi; nếu Kaggle dừng cả session thì xuất ZIP không được bảo đảm. Không tự sửa trạng thái thành complete. Để chạy lại, đổi cả experiment_id/output; để export lại chọn thư mục EXPORT_DIR mới.

In [6]:
try:
    helper("run", "--max-hours", MAX_HOURS)
finally:
    helper("export", "--export-dir", EXPORT_DIR)

print("Download from Kaggle Output:", EXPORT_DIR)


Running: scripts/tiny_imagenetc.py --config /kaggle/working/streaming-tta-tiny-state-memory/configs/kaggle_tiny_preliminary.yaml --execute --max-hours 10.0
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth

  0%|          | 0.00/97.8M [00:00<?, ?B/s]
  5%|▌         | 5.12M/97.8M [00:00<00:01, 52.3MB/s]
 10%|█         | 10.1M/97.8M [00:00<00:01, 50.0MB/s]
 28%|██▊       | 27.6M/97.8M [00:00<00:00, 109MB/s] 
 46%|████▌     | 45.0M/97.8M [00:00<00:00, 137MB/s]
 64%|██████▍   | 62.6M/97.8M [00:00<00:00, 154MB/s]
 82%|████████▏ | 80.4M/97.8M [00:00<00:00, 165MB/s]
100%|██████████| 97.8M/97.8M [00:00<00:00, 144MB/s]
tiny_pilot_seed101 gaussian_noise+brightness__to__defocus_blur source completed
tiny_pilot_seed101 gaussian_noise+brightness__to__defocus_blur norm completed
tiny_pilot_seed101 gaussian_noise+brightness__to__defocus_blur tent completed
tiny_pilot_seed101 gaussian_noise+brightness__to__defocus_blur 

## 6. Đọc W16 và accuracy

Q=512: một quyết định khác tương ứng 0.1953125 điểm phần trăm. Giữ từng kịch bản riêng khi xem bảng. Disagreement không tự biểu thị lợi/hại, và bằng 0 không đảm bảo logits/loss giống nhau.

Hai CSV chính nằm trong results: `preliminary_history.csv`, `preliminary_performance.csv`. ZIP gồm source anchor, mapping/projected indices, NPZ, manifests, logs, bảng/hình; không gồm ảnh Input. Kết quả chỉ thuộc **Tiny exploratory transfer**, không so trực tiếp với số Stage A/Stage B ImageNet-C.

In [7]:
import pandas as pd
import yaml
from IPython.display import display, FileLink

cfg = yaml.safe_load(CONFIG.read_text())
RESULTS = REPO / cfg["output"]
status = json.loads((RESULTS / "run_status.json").read_text())
audit = json.loads((RESULTS / "evaluation_audit.json").read_text())
controls = json.loads((RESULTS / "controls.json").read_text())
assert status["status"] == "complete" and audit["passed"]
assert controls and all(c["passed"] for c in controls)
print("Run:", status)
print("Audit:", audit["counts"])
history = pd.read_csv(RESULTS / "preliminary_history.csv")
display(history[history.washout_batches == 16])
performance = pd.read_csv(RESULTS / "preliminary_performance.csv")
display(performance[(performance.washout_batches == 16) & (performance.intervention == "none")])
archive = EXPORT_DIR / (cfg["experiment_id"] + "_artifacts.zip")
print("ZIP:", archive)
display(FileLink(str(archive)))


Run: {'status': 'complete', 'stage': 'TINY', 'seconds': 1608.3985719490001, 'controls_passed': 210, 'controls_total': 210, 'interpretation': 'Exploratory Tiny ImageNet-C, one panel; ImageNet-pretrained ResNet50 projected to 200 dataset synsets, input resized to 224. Not a Tiny-trained baseline or Stage B ImageNet-C result.'}
Audit: {'run_rows': 144, 'paired_rows': 72, 'predictions_audited': 144, 'pairs_audited': 72, 'scalar_comparisons': 1368, 'manifest_checks': 144, 'unique_original_ids': 512, 'panel_scenario_groups': 3}


,panel_id,scenario,method,washout_batches,intervention,n,disagreement,absolute_error_gap,nll_gap,first_batch_max_logit_diff,disagreement_pp
4,tiny_pilot_seed101,gaussian_noise+brightness__to__defocus_blur,source,16,none,512,0.000000,0.000000,0.000000,0.000000,0.000000
5,tiny_pilot_seed101,gaussian_noise+brightness__to__defocus_blur,source,16,all,512,0.000000,0.000000,0.000000,0.000000,0.000000
10,tiny_pilot_seed101,gaussian_noise+brightness__to__defocus_blur,norm,16,none,512,0.000000,0.000000,0.000000,0.000000,0.000000
11,tiny_pilot_seed101,gaussian_noise+brightness__to__defocus_blur,norm,16,all,512,0.000000,0.000000,0.000000,0.000000,0.000000
16,tiny_pilot_seed101,gaussian_noise+brightness__to__defocus_blur,tent,16,none,512,0.085938,0.005859,0.002793,0.317001,8.593750
17,tiny_pilot_seed101,gaussian_noise+brightness__to__defocus_blur,tent,16,all,512,0.000000,0.000000,0.000000,0.000000,0.000000
22,tiny_pilot_seed101,gaussian_noise+brightness__to__defocus_blur,sar_complete,16,none,512,0.000000,0.000000,0.000000,0.000000,0.000000
23,tiny_pilot_seed101,gaussian_noise+brightness__to__defocus_blur,sar_complete,16,all,512,0.000000,0.000000,0.000000,0.000000,0.000000
28,tiny_pilot_seed101,defocus_blur+gaussian_noise__to__brightness,source,16,none,512,0.000000,0.000000,0.000000,0.000000,0.000000
29,tiny_pilot_seed101,defocus_blur+gaussian_noise__to__brightness,source,16,all,512,0.000000,0.000000,0.000000,0.000000,0.000000


,scenario,method,washout_batches,intervention,accuracy,nll,brier,ece15,accuracy_percent
5,brightness+defocus_blur__to__gaussian_noise,norm,16,none,0.060547,5.029103,1.002339,0.090400,6.054688
11,brightness+defocus_blur__to__gaussian_noise,sar_complete,16,none,0.060547,5.022064,1.004588,0.093735,6.054688
17,brightness+defocus_blur__to__gaussian_noise,source,16,none,0.070312,5.541301,1.085905,0.262069,7.031250
23,brightness+defocus_blur__to__gaussian_noise,tent,16,none,0.066406,4.954335,1.003536,0.102278,6.640625
29,defocus_blur+gaussian_noise__to__brightness,norm,16,none,0.273438,3.298399,0.849415,0.057761,27.343750
35,defocus_blur+gaussian_noise__to__brightness,sar_complete,16,none,0.298828,3.122173,0.821354,0.067478,29.882812
41,defocus_blur+gaussian_noise__to__brightness,source,16,none,0.277344,3.577870,0.899980,0.171033,27.734375
47,defocus_blur+gaussian_noise__to__brightness,tent,16,none,0.303711,3.120101,0.823764,0.069602,30.371094
53,gaussian_noise+brightness__to__defocus_blur,norm,16,none,0.111328,4.636339,0.980201,0.101673,11.132812
59,gaussian_noise+brightness__to__defocus_blur,sar_complete,16,none,0.111328,4.607400,0.976098,0.102916,11.132812


ZIP: /kaggle/working/kaggle_tiny_exports/tiny_imagenetc_kaggle_preliminary_v1_artifacts.zip


/kaggle/working/kaggle_tiny_exports/tiny_imagenetc_kaggle_preliminary_v1_artifacts.zip